# Sprint 10 — Decision Engine Report

This notebook only loads retained inputs and calls `fpl_model.decision`; all decision and evaluation logic lives in `src`.

In [ ]:
from pathlib import Path

import pandas as pd

from fpl_model.decision import (
    PredictionColumns,
    PublicTeamLoader,
    decision_regret_backtest,
    decision_summary,
    make_decision,
)

In [ ]:
# Point these parameters at the promoted, retained prediction artifact.
TEAM_ID = 1
TARGET_GAMEWEEK = 1
PREDICTIONS_PATH = Path("artifacts/champion_predictions.parquet")
MODEL_ARTIFACT_ID = "replace-with-promoted-model-artifact-id"
SNAPSHOT_ARTIFACT_ID = "replace-with-deadline-snapshot-artifact-id"

In [ ]:
predictions = pd.read_parquet(PREDICTIONS_PATH)
team = PublicTeamLoader().load(TEAM_ID, TARGET_GAMEWEEK)
columns = PredictionColumns(
    position="position_at_deadline",
    club_id="team_id_at_deadline",
    xpts="xpts_direct_ensemble",
    p_play_any="p_play_any_ensemble",
    p_minutes_60="p_minutes_60_ensemble",
    expected_minutes="expected_minutes_ensemble",
)
decision = make_decision(
    team,
    predictions,
    columns=columns,
    snapshot_artifact_id=SNAPSHOT_ARTIFACT_ID,
    model_artifact_id=MODEL_ARTIFACT_ID,
)
decision_summary(decision)

In [ ]:
# Historical evaluation expects one complete 15-player squad per squad/GW.
historical = predictions.query("actual_points_gw.notna()").copy()
regret = decision_regret_backtest(
    historical,
    champion_prediction=columns.xpts,
    baselines={"last5": "pred_last5_points"},
    columns=columns,
)
regret.summary